# LEVIR-CD Dataset

[LEVIR-CD](https://chenhao.in/LEVIR/) (Large-scale EarthVision Remote Sensing Image Change Detection) is the most widely used benchmark for building change detection. It contains 637 bitemporal image pairs (1024×1024 px) of Google Earth imagery from Texas, USA, spanning 2002–2018.

**Ground truth**: Binary masks indicating building construction and demolition.

**Standard split**: 445 train / 64 val / 128 test pairs (at full 1024×1024 resolution). Open-CD works with 256×256 px chips chipped from these.

## References
- LEVIR-CD dataset: https://chenhao.in/LEVIR/
- Open-CD LEVIR-CD config: https://github.com/likyoo/open-cd/tree/main/configs

## 1. Download LEVIR-CD

The full dataset is ~5 GB. We download a small subset for demonstration.

In [ ]:
import os
import urllib.request
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# LEVIR-CD mini subset (publicly available sample)
sample_url = "https://github.com/likyoo/open-cd/releases/download/v1.0.0/LEVIR-CD-256-mini.zip"
dataset_dir = "LEVIR-CD-mini"

if not os.path.exists(dataset_dir):
    print("Downloading LEVIR-CD mini subset...")
    try:
        urllib.request.urlretrieve(sample_url, "LEVIR-CD-256-mini.zip")
        with zipfile.ZipFile("LEVIR-CD-256-mini.zip", "r") as zf:
            zf.extractall(dataset_dir)
        os.remove("LEVIR-CD-256-mini.zip")
        print("Downloaded and extracted.")
    except Exception as e:
        print(f"Download failed ({e}). Generating synthetic data for demonstration.")
        # Create synthetic LEVIR-CD structure for offline demonstration
        for split in ["train", "val"]:
            for subdir in ["A", "B", "label"]:
                os.makedirs(os.path.join(dataset_dir, split, subdir), exist_ok=True)
        # Generate synthetic image pairs
        for split in ["train", "val"]:
            n = 20 if split == "train" else 5
            for i in range(n):
                fname = f"{i:04d}.png"
                img_a = np.random.randint(50, 200, (256, 256, 3), dtype=np.uint8)
                img_b = img_a.copy()
                label = np.zeros((256, 256), dtype=np.uint8)
                # Add a "new building" region
                r, c = np.random.randint(50, 180), np.random.randint(50, 180)
                img_b[r:r+30, c:c+30] = [180, 180, 180]
                label[r:r+30, c:c+30] = 255
                Image.fromarray(img_a).save(os.path.join(dataset_dir, split, "A", fname))
                Image.fromarray(img_b).save(os.path.join(dataset_dir, split, "B", fname))
                Image.fromarray(label).save(os.path.join(dataset_dir, split, "label", fname))
        print("Synthetic dataset created.")
else:
    print(f"Dataset already exists at {dataset_dir}/")

## 2. Explore the Dataset Structure

In [ ]:
for split in ["train", "val"]:
    split_dir = os.path.join(dataset_dir, split)
    if os.path.exists(split_dir):
        n_files = len(os.listdir(os.path.join(split_dir, "A")))
        print(f"{split}: {n_files} image pairs")
    else:
        print(f"{split}: not found")

print("\nDirectory structure:")
print("  LEVIR-CD-mini/")
print("  ├── train/")
print("  │   ├── A/        # T1 (before) images")
print("  │   ├── B/        # T2 (after) images")
print("  │   └── label/    # binary change masks (255=changed, 0=unchanged)")
print("  └── val/")
print("      ├── A/")
print("      ├── B/")
print("      └── label/")

## 3. Visualize Bitemporal Pairs

In [ ]:
import random

train_a_dir = os.path.join(dataset_dir, "train", "A")
sample_files = random.sample(os.listdir(train_a_dir), min(3, len(os.listdir(train_a_dir))))

fig, axes = plt.subplots(len(sample_files), 3, figsize=(12, 4 * len(sample_files)))
if len(sample_files) == 1:
    axes = axes[np.newaxis, :]

for row, fname in enumerate(sample_files):
    img_a = np.array(Image.open(os.path.join(dataset_dir, "train", "A", fname)))
    img_b = np.array(Image.open(os.path.join(dataset_dir, "train", "B", fname)))
    label = np.array(Image.open(os.path.join(dataset_dir, "train", "label", fname)))

    axes[row, 0].imshow(img_a)
    axes[row, 0].set_title(f"T1 (before)\n{fname}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(img_b)
    axes[row, 1].set_title("T2 (after)")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(label, cmap="Reds")
    changed_pct = np.sum(label > 0) / label.size * 100
    axes[row, 2].set_title(f"Change mask ({changed_pct:.1f}% changed)")
    axes[row, 2].axis("off")

plt.suptitle("LEVIR-CD sample pairs", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Dataset Statistics

In [ ]:
label_dir = os.path.join(dataset_dir, "train", "label")
change_ratios = []

for fname in os.listdir(label_dir):
    if fname.endswith(".png"):
        label = np.array(Image.open(os.path.join(label_dir, fname)))
        change_ratios.append(np.sum(label > 0) / label.size)

print(f"Training pairs: {len(change_ratios)}")
print(f"Mean change ratio:   {np.mean(change_ratios)*100:.2f}%")
print(f"Median change ratio: {np.median(change_ratios)*100:.2f}%")
print(f"Max change ratio:    {np.max(change_ratios)*100:.2f}%")
print(f"\nClass imbalance: {(1-np.mean(change_ratios))/np.mean(change_ratios):.0f}:1 (unchanged:changed)")
print("\nTip: For imbalanced datasets, use weighted BCE loss or Dice loss in training.")

plt.figure(figsize=(7, 4))
plt.hist(np.array(change_ratios) * 100, bins=20, edgecolor="black")
plt.xlabel("% changed pixels per image")
plt.ylabel("Count")
plt.title("Distribution of change ratio across training samples")
plt.tight_layout()
plt.show()

## 5. Open-CD LEVIRCDDataset Integration

In [ ]:
from opencd.datasets import LEVIRCDDataset
from mmengine.dataset import BaseDataset

# Open-CD provides a ready-to-use dataset class for LEVIR-CD
print("Open-CD LEVIR-CD dataset class:")
print(f"  {LEVIRCDDataset}")
print(f"\nDefault metainfo: {LEVIRCDDataset.METAINFO}")
print("\nConfigure in a config file as:")
print("""
dataset_type = 'LEVIRCDDataset'
data_root = 'data/LEVIR-CD256/'

train_dataloader = dict(
    batch_size=8,
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        data_prefix=dict(img_path='train/A/', img_path2='train/B/', seg_map_path='train/label/'),
        pipeline=train_pipeline,
    )
)
""")